# 02 - Exploración de la variable objetivo

Este notebook parte de la base consolidada (`data/processed/graduados_2020_2025.csv`) generada en `01_exploracion_graduados.ipynb` y se enfoca en la **variable que el modelo va a predecir**: la proporción de mujeres entre los graduados de cada programa.

Objetivos:
1. Definir la unidad de análisis del modelo (programa-año).
2. Describir la distribución de la variable objetivo y el efecto de los programas pequeños.
3. Medir cuánto explica cada variable académica, institucional y geográfica (eta²).
4. Revisar dispersión, interacciones y estabilidad en el tiempo (riesgo de fuga de información).

## 1. Carga de la base procesada

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/graduados_2020_2025.csv")
print(df.shape)
print(df.columns.tolist())
print(df["sexo"].value_counts(dropna=False))

## 2. Unidad de análisis: programa-año

Cada fila de la base original es programa + sexo + semestre. Para el modelo se agrega a una fila por **programa - institución - departamento - nivel - año**, con el número de mujeres, hombres, total y la proporción de mujeres (variable objetivo).

In [ ]:
# Usar la versión normalizada de la columna si existe
def col(nombre):
    return f"{nombre}_normalizado" if f"{nombre}_normalizado" in df.columns else nombre

llaves = ["anio", "institucion", col("sector"), "caracter_institucion",
          col("departamento"), "programa", col("nivel_formacion"), "area_conocimiento"]

sexo = df["sexo"].astype(str).str.upper().str.strip()
df["mujeres"] = np.where(sexo.str.startswith(("MUJER", "FEMEN")), df["graduados"], 0)
df["hombres"] = np.where(sexo.str.startswith(("HOMBRE", "MASCU")), df["graduados"], 0)

prog = df.groupby(llaves, dropna=False)[["mujeres", "hombres"]].sum().reset_index()
prog["total"] = prog["mujeres"] + prog["hombres"]
prog = prog[prog["total"] > 0].copy()
prog["prop_mujeres"] = prog["mujeres"] / prog["total"]

print(prog.shape)
prog["prop_mujeres"].describe()

## 3. Distribución de la variable objetivo y tamaño del programa

Un programa con muy pocos graduados puede dar 0 % o 100 % de mujeres por azar. Esta sección cuantifica cuántos casos así existen, para decidir si el modelo filtra por un mínimo de graduados o pondera por el total.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(prog["prop_mujeres"], bins=40)
ax[0].set_title("Proporción de mujeres por programa-año")
ax[1].hist(np.log10(prog["total"]), bins=40)
ax[1].set_title("Graduados por programa-año (log10)")
plt.tight_layout()
plt.show()

for corte in [5, 10, 30]:
    print(f"Programas-año con menos de {corte} graduados: {(prog['total'] < corte).mean():.1%}")

print(f"Con 0% mujeres:   {(prog['prop_mujeres'] == 0).mean():.1%}")
print(f"Con 100% mujeres: {(prog['prop_mujeres'] == 1).mean():.1%}")
print(f"Media simple:     {prog['prop_mujeres'].mean():.2%}")
print(f"Media ponderada:  {prog['mujeres'].sum() / prog['total'].sum():.2%}")

## 4. ¿Cuánto explica cada variable? (eta² ponderado)

El eta² indica qué porcentaje de la variación de la proporción de mujeres entre programas se explica por cada variable por sí sola. Responde directamente el "en qué medida" de la pregunta del proyecto. Nota: `institucion` tiene muchas categorías, por lo que su eta² tiende a estar inflado.

In [ ]:
def eta2(data, var, y="prop_mujeres", w="total"):
    d = data.dropna(subset=[var]).copy()
    d["yw"] = d[y] * d[w]
    media = d["yw"].sum() / d[w].sum()
    g = d.groupby(var).agg(yw=("yw", "sum"), w=(w, "sum"))
    g["m"] = g["yw"] / g["w"]
    ss_entre = (g["w"] * (g["m"] - media) ** 2).sum()
    ss_total = (d[w] * (d[y] - media) ** 2).sum()
    return ss_entre / ss_total

variables = ["area_conocimiento", col("nivel_formacion"), col("sector"),
             "caracter_institucion", col("departamento"), "anio", "institucion"]

res = pd.Series({v: eta2(prog, v) for v in variables}).sort_values()
print(res.round(3))

res.plot(kind="barh", figsize=(8, 4), title="Varianza explicada por cada variable (eta², ponderado)")
plt.show()

## 5. Dispersión dentro de cada grupo

Los promedios por grupo pueden esconder mucha variación interna. Se usan programas con 10 o más graduados para evitar el ruido de los programas pequeños.

In [ ]:
base = prog[prog["total"] >= 10]

for var in ["area_conocimiento", col("nivel_formacion"), col("sector"), "caracter_institucion"]:
    orden = base.groupby(var)["prop_mujeres"].median().sort_values().index
    datos = [base.loc[base[var] == k, "prop_mujeres"] for k in orden]
    plt.figure(figsize=(10, 0.45 * len(orden) + 1.5))
    plt.boxplot(datos, vert=False)
    plt.yticks(range(1, len(orden) + 1), orden)
    plt.axvline(0.5, linestyle="--", color="grey")
    plt.title(f"Proporción de mujeres por programa según {var} (10+ graduados)")
    plt.tight_layout()
    plt.show()

## 6. Interacción área x nivel de formación

Revisa si el efecto del nivel de formación (por ejemplo, la caída en doctorado) ocurre en todas las áreas o solo en algunas. Si depende del área, se justifica un modelo que capture interacciones (árboles, boosting).

In [ ]:
t = prog.groupby(["area_conocimiento", col("nivel_formacion")])[["mujeres", "total"]].sum()
tabla = (t["mujeres"] / t["total"]).unstack()

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(tabla, cmap="RdBu", vmin=0.2, vmax=0.8, aspect="auto")
ax.set_xticks(range(len(tabla.columns)), tabla.columns, rotation=45, ha="right")
ax.set_yticks(range(len(tabla.index)), tabla.index)
for i in range(tabla.shape[0]):
    for j in range(tabla.shape[1]):
        v = tabla.iloc[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="Proporción de mujeres")
plt.title("Proporción de mujeres por área y nivel de formación")
plt.tight_layout()
plt.show()

## 7. Estabilidad en el tiempo y riesgo de fuga de información

Si la proporción de un programa en el año t está muy correlacionada con la del año t-1, el historial del programa predice casi todo. Esto obliga a usar validación temporal (entrenar con años anteriores y probar con los más recientes). Al final se guarda la base programa-año para la etapa de modelado.

In [ ]:
id_prog = [c for c in llaves if c != "anio"]
prog = prog.sort_values(id_prog + ["anio"])
prog["prop_anio_anterior"] = prog.groupby(id_prog, dropna=False)["prop_mujeres"].shift(1)

p = prog.dropna(subset=["prop_anio_anterior"])
p = p[p["total"] >= 10]
print(f"Correlación año t vs año t-1: {p['prop_mujeres'].corr(p['prop_anio_anterior']):.3f}")

anios_por_prog = prog.groupby(id_prog, dropna=False)["anio"].nunique()
print(anios_por_prog.value_counts().sort_index())

plt.scatter(p["prop_anio_anterior"], p["prop_mujeres"], s=5, alpha=0.2)
plt.xlabel("Proporción año anterior")
plt.ylabel("Proporción año actual")
plt.title("Persistencia de la proporción de mujeres por programa")
plt.show()

prog.to_csv("../data/processed/programas_anio.csv", index=False, encoding="utf-8-sig")

## 8. Conclusiones

*Pendiente: completar después de ejecutar el notebook, conectando cada hallazgo con la pregunta de negocio.*